In [9]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
import random
from tqdm.auto import tqdm
import pandas as pd
import itertools
from statistics import mode

In [10]:
val_data = []
with open("dev.jsonl", "r") as f:
    data = f.readlines()

for line in data:
    val_data.append(json.loads(line))


test_data = []
with open("test.jsonl", "r") as f:
    data = f.readlines()

for line in data:
    test_data.append(json.loads(line))

In [25]:
test_data[0].keys()

dict_keys(['id', 'question', 'formatted_question'])

In [11]:

BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
# torch.set_default_device(device)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("logs/qasc_qwen3B/model", trust_remote_code=True, device_map="auto")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [12]:
print(  val_data[0].keys(), )
print(val_data[0]['question']['choices'], val_data[0]['question']['stem'])
# print(  test_data[0]['question']['choices'][3], )


# print(test_data[0].keys())


dict_keys(['id', 'question', 'answerKey', 'fact1', 'fact2', 'combinedfact', 'formatted_question'])
[{'text': 'sand', 'label': 'A', 'para': 'Generally if there is a beach on the shore, it is beautiful sand. What sand there is, is liberally peppered with seaweed. Faith is to the human what sand is to the ostrich. Sun, Sand and a perfect climate contribute to a lively youthful atmosphere. Simply described, a sand tray is a small sand box designed for indoor use. Blast Describes a shot from a sand bunker. Also the term sand is used interchangeably. Hookworms are often found in the soil or sand in moderate climates. What sand remains is but residue. Generally such soils are sands or loamy sands.'}, {'text': 'occurs over a wide range', 'label': 'B', 'para': "Engineering is a wide range of activities that can be described best in terms of functions. Climate A wide range of climatic conditions are present in the large geographical range of redbud. Aroids grow all over the world and occur in a 

In [13]:
from string import Template
import string
prompt_template= Template('''Instruct: Answer the following question using the context provided, reason over it because only one of the context is relevant . Please generate only answer choice (1, 2, 3, 4, 5, 6, 7 or 8) without any explanations\n
$question
context: $context      
                                                        
$options
$question
''')


In [14]:
# Function to clean and split text into words
def preprocess(text):
    
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return set(text.split())


def choose_most_likely_option(options, candidate_answer):
    candidate_words = preprocess(candidate_answer)
    
    best_option = None
    max_overlap = 0
    
    for option in options:
        option_words = preprocess(option)
        overlap = len(candidate_words.intersection(option_words))
        
  
        if overlap > max_overlap:
            max_overlap = overlap
            best_option = option
            
    return best_option, options.index(best_option)+1

In [15]:
def answer_reggexer(text,options, prompt_len):
    answer_only = text[prompt_len:]
    try:
        id = int(answer_only[0])


    except:
        try:
        # print(answer_only)
            _, id =  choose_most_likely_option(options, answer_only)
        except:


            id  = random.randint(1,8)
    return id
    

In [16]:

# data[0]
submission = {"answers":[]}
option_header = ["option 1 ", "option 2 ", "option 3 ", "option 4 ", "option 5 ","option 6 ", "option 7 ", "option 8 " ]

pbar = tqdm(range(len(val_data)))
k = 20
for example in val_data:
    options = []
    opts = []
    context = ''
    for i, sample in enumerate(example['question']['choices']):
        # print(key)
        
    
        options.append(option_header[i]+ sample['text'])
        opts.append((sample['text'], option_header[i].split("option")[1]))
        context += '\n'+ sample['para']

    combined_fact = example['combinedfact']

    all_permutations = list(itertools.permutations(opts))
    all_permutations = random.sample(all_permutations, k if len(all_permutations)>k else len(all_permutations))
    # print(all_permutations)
    batch_prompts = []
    batch_options = []
    option_maps = []
    for option_set in all_permutations:
        options  = []
        option_map = {}
        for i in range(len(option_header)):
            options.append(option_header[i] + option_set[i][0])
            option_map[i+1]  = option_set[i][1]
        option_maps.append(option_map)
        batch_options.append(options)
        batch_prompts.append(prompt_template.substitute( question  = example['question']['stem'], context=  combined_fact, options = "\n".join(options))+ "option ")

    # prompt_sample = prompt_template.substitute( question  = example['question']['stem'], context=  combined_fact, options = "\n".join(options))

    # print(prompt_sample)

    input  = tokenizer(batch_prompts, padding="longest", return_tensors="pt").to("cuda")
    # print(input)
    out = model.generate(**input,  max_new_tokens=50)
    texts = tokenizer.batch_decode(out,skip_special_tokens=True)
    answers = []
    for text, options, option_map in zip(texts, batch_options, option_maps):
       
        id = answer_reggexer(text,options, len(batch_prompts[0]))
        answers.append(int(option_map[id]))


    
    submission["answers"].append(mode(answers))
    pd.DataFrame(submission).to_csv("qwenk"+str(k)+"_qasc_combined_fact_qmos.csv")
    
    pbar.set_description(f"Pred: {id:.4f}")
    pbar.update(1)
pbar.close()
# print(answer_only)

  0%|          | 0/926 [00:00<?, ?it/s]